In [8]:
import pandas as pd, numpy as np, os
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Phase5"
os.makedirs(BASE, exist_ok=True)
raw = pd.read_csv(f"{BASE}/SCMS_Delivery_History_Dataset(ORIGINAL).csv")
df = raw.copy()
print(raw.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(10324, 33)


In [9]:
df.columns = df.columns.str.strip()
print(df.duplicated().sum())
print(df["ID"].duplicated().sum())
v = df["Vendor"]
print(v.nunique())
print(v.str.strip().str.upper().nunique())
for c in ["Line Item Value",
          "Line Item Quantity",
          "Weight (Kilograms)"]:
    n = pd.to_numeric(df[c], errors="coerce")
    print(c, df[c].dtype, n.isna().sum())
print(df["Line Item Value"].describe())

0
0
73
73
Line Item Value float64 0
Line Item Quantity int64 0
Weight (Kilograms) object 3952
count    1.032400e+04
mean     1.576506e+05
std      3.452921e+05
min      0.000000e+00
25%      4.314593e+03
50%      3.047147e+04
75%      1.664471e+05
max      5.951990e+06
Name: Line Item Value, dtype: float64


In [10]:
# Clean vendors, parse dates, add late and RDC flags, make value numeric, drop text-heavy columns
fmt = "%d-%b-%y"
df["Vendor"] = (df["Vendor"].str.strip()
    .str.replace(r"\s+", " ", regex=True))
df["sched"] = pd.to_datetime(
    df["Scheduled Delivery Date"], format=fmt)
df["deliv"] = pd.to_datetime(
    df["Delivered to Client Date"], format=fmt)
df["days_late"] = (df["deliv"]
    - df["sched"]).dt.days
df["late"] = df["days_late"] > 0
df["is_rdc"] = df["Vendor"] == "SCMS from RDC"
df["value"] = pd.to_numeric(
    df["Line Item Value"], errors="coerce")
df = df.drop(columns=["Freight Cost (USD)", "Weight (Kilograms)"])

In [11]:
assert len(df) == 10324, "rows changed"
assert df["days_late"].notna().all(), "bad dates"
assert df["value"].notna().all(), "bad values"
assert df["PO / SO #"].nunique() == 6233, "PO count"
print(round(1 - df["late"].mean(), 3))
print((df["days_late"] < -90).sum(),
      (df["days_late"] > 90).sum())
vc = df.loc[~df["is_rdc"], "Vendor"].value_counts()
print((vc >= 20).sum())

0.885
224 40
24


In [12]:
df.to_csv(f"{BASE}/scms_clean.csv", index=False)
print(os.listdir(BASE))

['SCMS_Delivery_History_Dataset(ORIGINAL).csv', 'brief.gdoc', 'cleaning log.gdoc', 'scms_clean.csv']


Cleaning log: SCMS Delivery History (10,324 rows, 33 columns)

1. Raw data kept untouched in `raw`; all work done on a copy (`df`).
2. Duplicates: 0 duplicate rows and 0 duplicate IDs found. No action needed.
3. Vendor names: 70 lines had a double space in one vendor name (Shanghai Kehua Bioengineering Co.,Ltd (KHB)). Collapsed to single spaces. Vendor count stayed 73 before and after, so no names were merged and there were no spelling variants
4. Dates: Scheduled and Delivered dates were text (d-Mon-yy). Converted both with an explicit format; 0 failed.
5. Line Item Value: already numeric, 0 text entries. 17 lines have a value of 0; kept because they are real lines. They count in lateness but add $0 to spend at risk.
6. Freight Cost (USD) dropped: 4,126 text entries (40%), e.g. "Freight Included in Commodity Cost". Not needed for lateness.
7. Weight (Kilograms) dropped: 3,952 text entries (38%), e.g. "Weight Captured Separately". Not needed for lateness.
8. Outliers kept: 224 lines delivered more than 90 days early and 40 more than 90 days late. Early counts as on time,
   and late days will be reported with the median as well as the mean.
9.  SCMS from RDC flagged with is_rdc: 5,404 lines (52%). The count matches Fulfill Via = "From RDC" (5,404), so these are lines shipped from the distribution centre, not from an outside supplier. Reported separately from supplier rankings.
10. Columns not used were left as is: Dosage Form (1,736 missing), Line Item Insurance (USD) (287 missing), PQ First Sent to Client Date (360 missing).
11. Reconciled with Day 121: 10,324 rows, 6,233 PO/SO numbers, on-time rate 0.885, 24 suppliers with 20+ lines.
12. Saved the clean data as scms_clean.csv in Phase5